In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *
from lib.kpi_processor.KPIReportEurope import *
from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = KPIReport.kpi_list + KPIEmissionSource.kpi_list

In [4]:
cursor.execute("DROP VIEW IF EXISTS Country_Yearly_KPI;")
cursor.execute("DROP VIEW IF EXISTS Customer_Yearly_KPI;")
conn.commit()


In [5]:
query = """
CREATE VIEW IF NOT EXISTS Country_Yearly_KPI AS
SELECT
    kd.Year,
    kc.Name AS Country,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
INNER JOIN KPI_Country kc ON kd.CustomerId = kc.CountryId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year, kd.CustomerId, kc.Name;"""
cursor.execute(query)

query = """
CREATE VIEW IF NOT EXISTS Customer_Yearly_KPI AS
SELECT
    kd.Year,
    kc.Name AS CustomerName,
    kc.Country AS Country,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
INNER JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year,kd.CustomerId, kc.Name;"""
cursor.execute(query)

conn.commit()

In [6]:
query = "SELECT * FROM Country_Yearly_KPI WHERE Year = 2026;"
query_customer = "SELECT * FROM Customer_Yearly_KPI WHERE Year = 2026;"
country_df = pd.read_sql_query(query, conn)
customer_df = pd.read_sql_query(query_customer, conn)
conn.close()
display(country_df)
display(customer_df)


,Year,Country,FOVMain,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,ServicePipeKm,ServicePipeCoveredKm,ReportCount,...,Bm2Density,NGDensity,PGDensity,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2026,United Kingdom,94.31,79435.57,74611.26,77516.83,73103.40,1918.74,1507.87,2478.0,...,0.13,1.10,0.18,20.00,1.02,68.82,10.15,76.05,12.72,11.23
1,2026,Italy,91.89,200558.85,184093.34,199462.50,183276.67,1096.35,816.67,4162.0,...,0.22,0.69,0.50,13.39,0.50,67.74,18.36,43.99,31.88,24.14
2,2026,Romania,94.66,183.72,174.11,162.63,153.94,21.10,20.17,7.0,...,0.49,1.11,1.68,17.08,1.65,63.58,17.70,38.88,58.52,2.61
3,2026,Switzerland,97.57,178.63,174.14,125.69,122.63,52.94,51.51,3.0,...,0.03,0.57,0.17,32.81,1.56,61.72,3.91,73.88,21.64,4.48
4,2026,Greece,99.36,16665.23,16564.50,14279.27,14187.86,2385.97,2376.64,404.0,...,0.27,0.78,0.50,10.75,0.42,68.06,20.77,46.73,30.00,23.26
5,2026,Germany,92.48,34089.17,31602.80,23325.63,21571.45,10763.60,10031.40,1477.0,...,0.04,0.10,0.15,16.00,0.93,67.62,15.45,17.95,26.94,55.12
6,2026,Poland,37.82,4127.24,1560.89,4127.24,1560.89,0.00,0.00,114.0,...,0.04,0.25,0.26,40.61,2.14,50.19,7.06,31.02,31.42,37.56
7,2026,Austria,39.64,185.12,65.20,146.00,57.87,39.13,7.33,42.0,...,0.03,0.05,0.21,0.00,0.00,88.24,11.76,7.89,36.84,55.26
8,2026,Ireland,95.27,4420.82,4190.69,4225.02,4025.13,195.81,165.56,131.0,...,0.02,0.13,0.05,26.69,1.36,63.14,8.81,38.10,15.46,46.44
9,2026,Netherlands,94.14,1073.16,1009.69,710.45,668.79,362.71,340.90,38.0,...,0.06,0.17,0.36,27.72,2.62,58.80,10.86,13.98,28.92,57.11


,Year,CustomerName,Country,FOVMain,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,ServicePipeKm,ServicePipeCoveredKm,...,Bm2Density,NGDensity,PGDensity,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2026,Wales and West Utilities,United Kingdom,92.99,2572.39,2392.00,2572.39,2392.00,0.00,0.00,...,0.18,1.57,0.40,22.97,1.29,66.53,9.22,74.39,18.94,6.66
1,2026,APRETIGAS,Italy,90.95,14783.39,13452.61,14498.15,13186.65,285.24,265.97,...,0.21,0.76,0.49,16.10,0.55,66.85,16.50,43.91,27.98,28.10
2,2026,ADRIGAS,Italy,95.54,774.91,739.87,765.20,731.09,9.71,8.78,...,0.08,0.44,0.43,29.68,2.78,58.73,8.81,50.85,49.15,0.00
3,2026,M Reti,Italy,99.94,84.40,84.35,84.40,84.35,0.00,0.00,...,0.57,1.74,1.77,12.50,0.00,71.28,16.22,32.81,33.26,33.93
4,2026,CPL CONCORDIA,Romania,94.66,183.72,174.11,162.63,153.94,21.10,20.17,...,0.49,1.11,1.68,17.08,1.65,63.58,17.70,38.88,58.52,2.61
5,2026,RETEGAS BARI spa,Italy,96.17,170.15,163.63,170.15,163.63,0.00,0.00,...,0.35,1.04,0.68,17.38,0.35,62.06,20.21,52.15,34.36,13.50
6,2026,ASTEA,Italy,95.36,177.86,169.61,177.86,169.61,0.00,0.00,...,0.14,0.95,0.48,23.05,1.65,65.43,9.88,62.40,31.78,5.81
7,2026,AIL,Switzerland,97.57,178.63,174.14,125.69,122.63,52.94,51.51,...,0.03,0.57,0.17,32.81,1.56,61.72,3.91,73.88,21.64,4.48
8,2026,Centria,Italy,98.68,212.17,209.37,212.17,209.37,0.00,0.00,...,0.14,0.62,0.35,10.40,0.00,75.25,14.36,43.43,24.58,31.99
9,2026,DEPA,Greece,99.36,16665.23,16564.50,14279.27,14187.86,2385.97,2376.64,...,0.27,0.78,0.50,10.75,0.42,68.06,20.77,46.73,30.00,23.26


In [7]:
KPIEmissionSource.kpi_list

['LisaCount',
 'LisaPSCount',
 'EmissionRate',
 'B0Count',
 'B1Count',
 'Bm1Count',
 'Bm2Count',
 'NGCount',
 'PGCount',
 'Not_NGCount',
 'EmissionRateLPM',
 'RepresentativeEmissionRate',
 'RepresentativeEmissionRateLPM',
 'B0RepEmissionRateLPM',
 'B1RepEmissionRateLPM',
 'Bm1RepEmissionRateLPM',
 'Bm2RepEmissionRateLPM',
 'B0RepEmissionRate',
 'B1RepEmissionRate',
 'Bm1RepEmissionRate',
 'Bm2RepEmissionRate',
 'LisaDensity',
 'InstantaneousEmission',
 'InstantaneousEmissionLPM',
 'InstantaneousRepEmission',
 'InstantaneousRepEmissionLPM',
 'InstantaneousRepEmissionB1',
 'InstantaneousRepEmissionB1LPM',
 'InstantaneousRepEmissionB0',
 'InstantaneousRepEmissionB0LPM',
 'InstantaneousRepEmissionBm1',
 'InstantaneousRepEmissionBm1LPM',
 'InstantaneousRepEmissionBm2',
 'InstantaneousRepEmissionBm2LPM',
 'B0Density',
 'B1Density',
 'Bm1Density',
 'Bm2Density',
 'NGDensity',
 'PGDensity',
 'B0Share',
 'B1Share',
 'Bm1Share',
 'Bm2Share',
 'NGShare',
 'PGShare',
 'Not_NGShare']